# VeritasCarbon - Instruction data generation (high-quality serial v3.0)

## CoDE framework – main ideas

### Core contributions (SIGMOD 2027)

1. **Meta-Expert**: Local Qwen2-72B generates per-chunk expert instructions; two-stage (Meta-Expert → Expert Agent); carbon-centric constraint.
2. **12 expert types (11 used in Phase 2)**: Base (QA, Summary, Extraction, Classification, Analysis), Analysis (Temporal, Benchmark, Greenwashing), Verification (Standard Alignment, Consistency; Promise Performance in Phase 3), Graph (Knowledge Graph).
3. **Adaptive expert selection**: 1–3 experts per chunk by length, topic, structure; layered selection (base required + optional advanced); rule + heuristic.
4. **CoDE collaboration**: Sequential experts (each output feeds the next); feedback loop (up to 2 rounds); quality score 0–1; early stop when above threshold.
5. **Domain knowledge**: ESG/GRI/SDG keywords; knowledge-base retrieval; knowledge in prompts.

### Technical notes

- **Local**: Qwen2-72B, no API cost.
- **Unsloth**: ~2–3× faster, 30–50% less VRAM.
- **Checkpoints**: Resume on interrupt.
- **Quality**: Auto evaluation + iteration, ~85–95%.

### Rough performance (RTX PRO 6000 96GB)

| Metric | Value |
|--------|--------|
| Per QA pair | 15–25 s |
| 1000 QA pairs | 4–7 h |
| Quality score | 85–95% |
| VRAM | 75–85 GB (no Unsloth) / 50–60 GB (Unsloth) |

---

### Important

- Do **not** edit this notebook while papermill is running it (output overwrites file).
- For background runs, save a copy as the output notebook (e.g. `02_InstructionGeneration_v3_output.ipynb`).


## 0. Environment check and paths


In [1]:
# Import unsloth before transformers for best performance
try:
    import unsloth
    UNSLOTH_AVAILABLE = True
except ImportError:
    UNSLOTH_AVAILABLE = False

import torch
import sys
from pathlib import Path

print("=" * 80)
print("Environment check")
print("=" * 80)
print()
print("[1. GPU]")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Count: {torch.cuda.device_count()}")
    print(f"  CUDA: {torch.version.cuda}")
    for i in range(torch.cuda.device_count()):
        total_mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"  GPU {i} VRAM: {total_mem:.1f}GB")
else:
    print("  GPU not available; will use CPU (slow).")
print()
MODEL_PATH = "/root/autodl-fs/models/Qwen2-72B-Instruct/Qwen/Qwen2-72B-Instruct"
print("[2. Model path]")
if Path(MODEL_PATH).exists():
    print(f"  OK: {MODEL_PATH}")
else:
    print(f"  Not found: {MODEL_PATH}")
print()
PROJECT_ROOT = Path("/root/autodl-fs/Veritas/VeritasCarbon_SIGMOD")
SRC_PATH = PROJECT_ROOT / "src"
print("[3. Project path]")
if SRC_PATH not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(SRC_PATH))
    print(f"  Added: {SRC_PATH}")
else:
    print("  src already in sys.path")
print()
print("[4. Modules]")
try:
    from instruction_generation.expert_selector_02_01 import ExpertSelector
    print("  ExpertSelector OK")
except ImportError as e:
    print(f"  ExpertSelector: {e}")
try:
    from instruction_generation.domain_knowledge_02_02 import DomainKnowledgeInjector
    print("  DomainKnowledgeInjector OK")
except ImportError as e:
    print(f"  DomainKnowledgeInjector: {e}")
try:
    from instruction_generation.coe_framework_02_03 import COEFramework, ExpertOutput
    print("  COEFramework OK")
except ImportError as e:
    print(f"  COEFramework: {e}")
try:
    from instruction_generation.expert_agents_02_04 import create_expert, BaseExpert
    print("  Expert Agents OK")
except ImportError as e:
    print(f"  Expert Agents: {e}")
print()
print("=" * 80)
print("Environment check done")
print("=" * 80)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/root/autodl-tmp/miniconda3/envs/qwen_unsloth/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
Environment check

[1. GPU]
  GPU available: NVIDIA RTX PRO 6000 Blackwell Server Edition
  GPU count: 1
  CUDA: 12.8
  GPU 0 VRAM: 95.0GB

[2. Model path]
  OK: /root/autodl-fs/models/Qwen2-72B-Instruct/Qwen/Qwen2-72B-Instruct

[3. Project path]
  Added src: /root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/src

[4. Key modules]
  ExpertSelector: importable
  DomainKnowledgeInjector: importable
  COEFramework: importable
  Expert Agents: importable

Environment check done


## 0.5 Network (optional; for Unsloth / Hugging Face Hub)

If you see `LocalEntryNotFoundError` (no access to Hugging Face Hub), run the cell below to configure.

1. **VPN** (easiest)
2. **Hugging Face mirror** (e.g. for mainland China)
3. **Standard mode** – run without Unsloth


In [2]:
# # Network config (Unsloth / Hugging Face Hub)
# # If LocalEntryNotFoundError: 1) VPN 2) HF mirror (uncomment below) 3) Skip, use standard mode
# # ============================================================================

# import os

# print("=" * 80)
# print("Network config")
# print("=" * 80)
# print()

# # ============================================================================
# # Unsloth key settings
# os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # comment if VPN works
# # os.environ['HTTPS_PROXY'] = 'http://127.0.0.1:7897'
# print('HF_ENDPOINT:', os.environ.get('HF_ENDPOINT', 'not set'))
# print('UNSLOTH_DISABLE_STATISTICS:', os.environ.get('UNSLOTH_DISABLE_STATISTICS', 'not set'))
# print("=" * 80)


## 1. Load Qwen2-72B (with Unsloth)


In [3]:
import torch
import sys
import os

print("=" * 80)
print("Load Qwen2-72B (Unsloth optional)")
print("=" * 80)
print()

# Python env
print("[Python env]")
print(f"  Python: {sys.executable}")
python_env = sys.executable
if "qwen_unsloth" in python_env:
    print("  qwen_unsloth env detected")
elif "conda" in python_env or "envs" in python_env:
    env_name = python_env.split("envs/")[-1].split("/")[0] if "envs" in python_env else "Unknown"
    print(f"  Env: {env_name}")
    print("  Prefer qwen_unsloth for Unsloth")
else:
    print("  Env: base")
    print("  Run: conda activate qwen_unsloth")
print()

MODEL_PATH = "/root/autodl-fs/models/Qwen2-72B-Instruct/Qwen/Qwen2-72B-Instruct"

print(f"Model path: {MODEL_PATH}")
print()

# Unsloth detection
USE_UNSLOTH = False
unsloth_error = None

print("[Unsloth]")
try:
    from unsloth import FastLanguageModel
    USE_UNSLOTH = True
    print("Unsloth detected, using acceleration (2-3x faster, 30-50% less VRAM)")
    print()
except ImportError as e:
    USE_UNSLOTH = False
    unsloth_error = str(e)
    print("Unsloth not detected; using standard mode")
    print()
    print("[Setup]")
    if "unsloth_zoo" in unsloth_error.lower():
        print("  Error: unsloth_zoo missing")
        print("  Fix: conda activate qwen_unsloth; pip install unsloth_zoo; re-run cell")
    else:
        print("  Error: Unsloth not installed")
        print("  Fix: conda activate qwen_unsloth; pip install unsloth unsloth_zoo; re-run cell")
    print()
    print("  You can skip Unsloth and use standard mode (slower)")
    print()
except Exception as e:
    USE_UNSLOTH = False
    unsloth_error = str(e)
    print(f"Unsloth check error: {e}")
    print("  Using standard mode")
    print()

# Load model
if USE_UNSLOTH:
    print("Loading model with Unsloth...")
    print()
    
    # Network config for Unsloth
    print("[Network for Unsloth]")
    
    # Disable Unsloth stats download (avoid LocalEntryNotFoundError)
    os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
    print("  UNSLOTH_DISABLE_STATISTICS=1")
    
    # Use HF mirror if VPN not in effect
    if not os.environ.get('HF_ENDPOINT'):
        os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
        print("  HF_ENDPOINT=https://hf-mirror.com")
    else:
        print(f"  HF_ENDPOINT set: {os.environ.get('HF_ENDPOINT')}")
    print()
    
    # Unsloth only, no fallback
    max_retries = 3
    retry_count = 0
    model_loaded = False
    
    while retry_count < max_retries and not model_loaded:
        try:
            if retry_count > 0:
                print(f"Retry {retry_count}/{max_retries - 1}...")
                import time
                time.sleep(2)
            else:
                print("Loading model...")
            print()
            
            # Load with Unsloth (4-bit to reduce VRAM)
            # 4-bit: ~91GB -> 40-50GB VRAM, <5% quality loss
            # device_map={"": 0} forces single-GPU to avoid CPU/Disk
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=MODEL_PATH,
                max_seq_length=2048,
                dtype=None,
                load_in_4bit=True,  # 4-bit for lower VRAM
                trust_remote_code=True,
                device_map={"": 0},
                local_files_only=False,
            )
            FastLanguageModel.for_inference(model)
            print()
            print("=" * 80)
            print("Unsloth model loaded.")
            print("=" * 80)
            model_loaded = True
            
        except Exception as e:
            retry_count += 1
            error_msg = str(e)
            error_type = type(e).__name__
            
            print(f"Load failed (attempt {retry_count}/{max_retries})")
            print(f"  Error: {error_type}")
            print(f"  Message: {error_msg[:300]}...")
            print()
            
            # Check for network-related error
            is_network_error = (
                "LocalEntryNotFoundError" in error_msg or
                "connection" in error_msg.lower() or
                "hub" in error_msg.lower() or
                "download" in error_msg.lower() or
                "internet" in error_msg.lower() or
                "network" in error_msg.lower()
            )
            
            if is_network_error:
                print("[Network - attempting fix]")
                
                os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
                print("  UNSLOTH_DISABLE_STATISTICS=1")
                
                # Set mirror
                if not os.environ.get('HF_ENDPOINT'):
                    os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
                    print("  HF_ENDPOINT=https://hf-mirror.com")
                
                if retry_count < max_retries:
                    print("  Retrying in 2s...")
                    print()
                else:
                    print()
                    print("=" * 80)
                    print("All retries failed")
                    print("=" * 80)
                    print()
                    print("[Final steps]")
                    print("1. Check VPN: ensure it is connected; test: curl https://huggingface.co")
                    print("2. If VPN needs proxy: os.environ['HTTPS_PROXY']='http://127.0.0.1:7890'; re-run cell")
                    print("3. Run network config cell (Cell 4): UNSLOTH_DISABLE_STATISTICS=1, HF_ENDPOINT set")
                    print("4. Check firewall if above fail")
                    raise RuntimeError(
                        "Unsloth load failed: network. Ensure VPN or mirror, re-run cell. No fallback to standard mode."
                    )
            else:
                # Not network; re-raise
                if retry_count < max_retries:
                    print("  Retrying in 2s...")
                    print()
                else:
                    raise RuntimeError(
                        f"Unsloth load failed (non-network): {error_type}: {error_msg[:200]}"
                    )
    
    if not model_loaded:
        raise RuntimeError("Unsloth load failed: max retries reached; no fallback to standard mode.")

if not USE_UNSLOTH:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True
    )
    print("Tokenizer loaded")
    print()
    
    print("Loading model (may take a few minutes)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    print("Model load done")

print()
print("[Model info]")
try:
    print(f"  - Params: {model.num_parameters() / 1e9:.1f}B")
except:
    print(f"  - Params: 72.0B (Qwen2-72B)")

# Quantization
quantization_config = None
if hasattr(model, 'config') and hasattr(model.config, 'quantization_config'):
    quantization_config = model.config.quantization_config
elif hasattr(model, 'quantization_config'):
    quantization_config = model.quantization_config

if quantization_config:
    quant_bits = getattr(quantization_config, 'bits', None)
    if quant_bits:
        print(f"  - Quant: {quant_bits}-bit")
    else:
        print("  - Quant: 4-bit")
else:
    print("  - Quant: none (bfloat16)")

print(f"  - dtype: {model.dtype}")
print(f"  - Mode: {'Unsloth (2-3x faster)' if USE_UNSLOTH else 'Standard'}")
if hasattr(model, 'hf_device_map'):
    print(f"  - device_map: {model.hf_device_map}")

if torch.cuda.is_available():
    print()
    print("[VRAM]")
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        reserved = torch.cuda.memory_reserved(i) / 1024**3
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        remaining = total - reserved
        usage_percent = (reserved / total) * 100
        
        print(f"  GPU {i}: {allocated:.1f}GB / {total:.1f}GB (allocated)")
        print(f"         {reserved:.1f}GB / {total:.1f}GB (reserved)")
        print(f"         Free: {remaining:.1f}GB ({100-usage_percent:.1f}%)")
        
        # VRAM note
        if remaining < 5:
            print("         Low VRAM; inference may need more (e.g. KV cache)")
        elif remaining < 10:
            print("         Consider 4-bit quantization")
        else:
            print("         VRAM OK for inference")

print()
print("=" * 80)
print("Model load done")
print("=" * 80)


Load Qwen2-72B (Unsloth optional)

[Python env]
  Python: /root/autodl-tmp/miniconda3/envs/qwen_unsloth/bin/python
  qwen_unsloth env detected

Model path: /root/autodl-fs/models/Qwen2-72B-Instruct/Qwen/Qwen2-72B-Instruct

[Unsloth]
Unsloth detected, using acceleration (2-3x faster, 30-50% less VRAM)

Loading model with Unsloth...

[Network for Unsloth]
  UNSLOTH_DISABLE_STATISTICS=1
  HF_ENDPOINT=https://hf-mirror.com

Loading model...

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downl

Loading checkpoint shards: 100%|██████████| 37/37 [10:04<00:00, 16.33s/it]



Unsloth model loaded.

[Model info]
  - Params: 72.7B
  - Quantization: 4-bit
  - dtype: torch.bfloat16
  - Optim: Unsloth (2-3x faster)
  - device_map: {'': 0}

[VRAM]
  GPU 0: 38.5GB / 95.0GB (allocated)
         39.4GB / 95.0GB (reserved)
         Free: 55.6GB (58.5%)
         VRAM OK for inference

Model load done


## 2. Initialize expert system components

### 2.1 Create 11 Phase 2 expert instances (local Qwen2-72B)


In [ ]:
from instruction_generation.expert_agents_02_04 import (
    QAExpert, SummaryExpert, ExtractionExpert, ClassificationExpert, AnalysisExpert,
    TemporalAnalysisExpert, BenchmarkComparisonExpert, GreenwashingDetectionExpert,
    StandardAlignmentExpert, KnowledgeGraphExpert, ConsistencyVerificationExpert,
    BaseExpert
)

print("=" * 80)
print("Initialize 11 Phase 2 expert instances")
print("=" * 80)
print()

# Adapter for local Qwen2-72B (Expert interface normally uses API)
class LocalQwenExpertAdapter(BaseExpert):
    """Adapter: local Qwen2-72B to Expert interface (no API)."""
    
    def __init__(self, name, local_model, local_tokenizer, temperature=0.7, max_tokens=1024):
        """Do not call super().__init__ to avoid API client."""
        self.name = name
        self.local_model = local_model
        self.local_tokenizer = local_tokenizer
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.api_provider = "local_qwen"
        self.model_name = "Qwen2-72B-Instruct"
        
    def _call_local_model(self, prompt, max_tokens=None, batch_prompts=None):
        """Call local Qwen2-72B (supports batch).
        
        Args:
            prompt: single prompt; max_tokens; batch_prompts: optional list (batch mode).
        Returns: str or list of str.
        """
        # Batch
        if batch_prompts is not None:
            return self._call_local_model_batch(batch_prompts, max_tokens)
        
        # Single prompt
        messages = [{"role": "user", "content": prompt}]
        text = self.local_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = self.local_tokenizer([text], return_tensors="pt").to(self.local_model.device)
        
        with torch.no_grad():
            generated_ids = self.local_model.generate(
                **model_inputs,
                max_new_tokens=max_tokens or self.max_tokens,
                temperature=self.temperature,
                top_p=0.9,
                do_sample=True
            )
        
        generated_ids = [
            output_ids[len(input_ids):] 
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        response = self.local_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        return response
    
    def _call_local_model_batch(self, batch_prompts, max_tokens=None):
        """Batch call local Qwen2-72B. Args: batch_prompts, max_tokens. Returns list of strings."""
        if not batch_prompts:
            return []
        
        # Messages
        batch_messages = [[{"role": "user", "content": p}] for p in batch_prompts]
        
        # Tokenize
        batch_texts = [
            self.local_tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=True
            )
            for msgs in batch_messages
        ]
        
        # Pad
        model_inputs = self.local_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(self.local_model.device)
        
        # Generate
        with torch.no_grad():
            generated_ids = self.local_model.generate(
                **model_inputs,
                max_new_tokens=max_tokens or self.max_tokens,
                temperature=self.temperature,
                top_p=0.9,
                do_sample=True,
                pad_token_id=self.local_tokenizer.eos_token_id
            )
        
        # Decode new tokens only
        generated_ids_clean = [
            output_ids[len(input_ids):] 
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        responses = self.local_tokenizer.batch_decode(
            generated_ids_clean,
            skip_special_tokens=True
        )
        
        return [r.strip() for r in responses]
    
    def get_prompt_template(self, chunk_text: str, context=None, dynamic_instruction=None):
        """BaseExpert get_prompt_template."""
        dynamic_inst = dynamic_instruction or (context.get("dynamic_instruction", "") if context else "")
        knowledge = context.get("domain_knowledge", "") if context else ""
        
        prompt = f"""You are an ESG expert. Generate a high-quality Q&A pair from the text below.

{dynamic_inst}

{knowledge}

Text:
{chunk_text}

Output format:
Instruction: [question or task]
Response: [detailed answer]"""
        return prompt
    
    def parse_response(self, response: str):
        """Parse response into instruction and answer (supports Instruction:/Response: or 指令/回答)."""
        from instruction_generation.coe_framework_02_03 import ExpertOutput
        
        instruction = ""
        answer = ""
        
        if "指令" in response and "回答" in response:
            parts = response.split("回答")
            if len(parts) >= 2:
                instruction_part = parts[0]
                if "指令" in instruction_part:
                    instruction = instruction_part.split("指令")[-1].split("：")[-1].split(":")[-1].strip().replace("\n", " ").strip()
                answer = parts[1].split("：")[-1].split(":")[-1].strip()
        
        if not instruction or not answer:
            lines = response.split("\n")
            for idx, ln in enumerate(lines):
                if "指令" in ln and not instruction:
                    instruction = ln.split("指令")[-1].split("：")[-1].split(":")[-1].strip()
                elif "回答" in ln and not answer:
                    answer = "\n".join(lines[idx+1:]).strip().split("：")[-1].split(":")[-1].strip()
                    break
        
        if not instruction or len(instruction) < 5:
            instruction = "Analyze the following ESG-related content"
        if not answer or len(answer) < 10:
            answer = response.strip()
        
        return ExpertOutput(
            expert_name=self.name,
            instruction=instruction.strip(),
            response=answer.strip(),
            quality_score=0.0,
            metadata={"expert_type": "local_qwen"}
        )
    
    def generate(self, chunk_text: str, context=None):
        """Generate using local model."""
        prompt = self.get_prompt_template(chunk_text, context)
        response = self._call_local_model(prompt)
        return self.parse_response(response)

# Per-expert subclasses
class LocalQAExpert(LocalQwenExpertAdapter):
    """QA expert (local Qwen2-72B)."""
    def __init__(self, local_model, local_tokenizer):
        super().__init__("QA Expert", local_model, local_tokenizer)
    
    def generate(self, chunk_text, context=None):
        from instruction_generation.coe_framework_02_03 import ExpertOutput
        
        dynamic_instruction = context.get("dynamic_instruction", "") if context else ""
        knowledge = context.get("domain_knowledge", "") if context else ""
        
        prompt = f"""You are an ESG QA expert. Generate one high-quality Q&A pair from the text below.

{dynamic_instruction}

Requirements:
1. Question should be specific, clear, and answerable from the text
2. Focus on ESG (environment, social, governance)
3. Prefer carbon emissions and environmental management content
4. Answer should be detailed, accurate, and grounded in the text

{knowledge}

Text:
{chunk_text}

Output format:
Question: [your question]
Answer: [your answer]"""
        
        response = self._call_local_model(prompt)
        
        lines = response.split("\\n")
        instruction = ""
        answer = ""
        for i, line in enumerate(lines):
            if line.startswith("Question:") or line.startswith("Question：") or line.startswith("问题：") or line.startswith("问题:"):
                instruction = line.replace("Question:", "").replace("Question：", "").replace("问题：", "").replace("问题:", "").strip()
            elif line.startswith("Answer:") or line.startswith("Answer：") or line.startswith("答案：") or line.startswith("答案:"):
                answer = "\\n".join(lines[i:]).replace("Answer:", "").replace("Answer：", "").replace("答案：", "").replace("答案:", "").strip()
                break
        
        return ExpertOutput(
            expert_name=self.name,
            instruction=instruction,
            response=answer,
            quality_score=0.0,
            metadata={"expert_type": "qa_expert"}
        )


class LocalSummaryExpert(LocalQwenExpertAdapter):
    """Summary expert."""
    def __init__(self, local_model, local_tokenizer):
        super().__init__("Summary Expert", local_model, local_tokenizer)
    
    def generate(self, chunk_text, context=None):
        from instruction_generation.coe_framework_02_03 import ExpertOutput
        dynamic_instruction = context.get("dynamic_instruction", "") if context else ""
        
        prompt = f"""You are an ESG summary expert. Generate a summary task for the text below.

{dynamic_instruction}

Text:
{chunk_text}

Output format:
Instruction: [summary task instruction]
Response: [detailed summary]"""
        
        response = self._call_local_model(prompt, max_tokens=2048)
        
        # Parse logic (same as QA expert)
        instruction = ""
        answer = ""
        
        if "指令" in response and "回答" in response:
            parts = response.split("回答")
            if len(parts) >= 2:
                instruction_part = parts[0]
                if "指令" in instruction_part:
                    instruction = instruction_part.split("指令")[-1].split("：")[-1].split(":")[-1].strip().replace("\n", " ").strip()
                answer = parts[1].split("：")[-1].split(":")[-1].strip().lstrip().strip()
        
        if not instruction or not answer:
            lines = response.split("\n")
            for i, line in enumerate(lines):
                if "指令" in line and not instruction:
                    instruction = line.split("指令")[-1].split("：")[-1].split(":")[-1].strip()
                    if not instruction and i + 1 < len(lines):
                        instruction = lines[i + 1].strip()
                elif "回答" in line and not answer:
                    answer = "\n".join(lines[i+1:]).strip().split("：")[-1].split(":")[-1].strip()
                    break
        
        if not instruction or len(instruction) < 5:
            instruction = "Summarize the following text"
        if not answer or len(answer) < 10:
            answer = response.strip()
            if instruction in answer:
                answer = answer.replace(instruction, "").strip()
        
        instruction = instruction.strip()
        answer = answer.strip()
        
        return ExpertOutput(
            expert_name=self.name,
            instruction=instruction,
            response=answer,
            quality_score=0.0,  # Re-evaluated by COE
            metadata={"expert_type": "summary_expert"}
        )


class LocalExtractionExpert(LocalQwenExpertAdapter):
    """Extraction expert."""
    def __init__(self, local_model, local_tokenizer):
        super().__init__("Extraction Expert", local_model, local_tokenizer)
    
    def generate(self, chunk_text, context=None):
        from instruction_generation.coe_framework_02_03 import ExpertOutput
        dynamic_instruction = context.get("dynamic_instruction", "") if context else ""
        
        prompt = f"""You are an ESG extraction expert. Extract key ESG metrics from the text below.

{dynamic_instruction}

Text:
{chunk_text}

Output format:
Instruction: [extraction task instruction]
Response: [extracted key information]"""
        
        response = self._call_local_model(prompt, max_tokens=2048)
        
        # Parse logic (same as QA expert)
        instruction = ""
        answer = ""
        
        if "指令" in response and "回答" in response:
            parts = response.split("回答")
            if len(parts) >= 2:
                instruction_part = parts[0]
                if "指令" in instruction_part:
                    instruction = instruction_part.split("指令")[-1].split("：")[-1].split(":")[-1].strip().replace("\n", " ").strip()
                answer = parts[1].split("：")[-1].split(":")[-1].strip().lstrip().strip()
        
        if not instruction or not answer:
            lines = response.split("\n")
            for i, line in enumerate(lines):
                if "指令" in line and not instruction:
                    instruction = line.split("指令")[-1].split("：")[-1].split(":")[-1].strip()
                    if not instruction and i + 1 < len(lines):
                        instruction = lines[i + 1].strip()
                elif "回答" in line and not answer:
                    answer = "\n".join(lines[i+1:]).strip().split("：")[-1].split(":")[-1].strip()
                    break
        
        if not instruction or len(instruction) < 5:
            instruction = "Extract key information from the text"
        if not answer or len(answer) < 10:
            answer = response.strip()
            if instruction in answer:
                answer = answer.replace(instruction, "").strip()
        
        instruction = instruction.strip()
        answer = answer.strip()
        
        return ExpertOutput(
            expert_name=self.name,
            instruction=instruction,
            response=answer,
            quality_score=0.0,  # Re-evaluated by COE
            metadata={"expert_type": "extraction_expert"}
        )


# ============================================================================
# Specialized subclasses for the remaining 8 experts
# ============================================================================

class LocalClassificationExpert(LocalQwenExpertAdapter):
    """Classification expert."""
    def __init__(self, local_model, local_tokenizer):
        super().__init__("Classification Expert", local_model, local_tokenizer)
    
    def get_prompt_template(self, chunk_text: str, context=None, dynamic_instruction=None):
        """Classification prompt template."""
        dynamic_inst = dynamic_instruction or (context.get("dynamic_instruction", "") if context else "")
        knowledge = context.get("domain_knowledge", "") if context else ""
        
        prompt = f"""You are an ESG classification expert. Classify and categorize ESG-related content in the text.

{dynamic_inst}

{knowledge}

Text:
{chunk_text}

Output format:
Instruction: [classification task instruction]
Response: [classification result and categories]"""
        return prompt


class LocalAnalysisExpert(LocalQwenExpertAdapter):
    """Analysis expert."""
    def __init__(self, local_model, local_tokenizer):
        super().__init__("Analysis Expert", local_model, local_tokenizer)
    
    def get_prompt_template(self, chunk_text: str, context=None, dynamic_instruction=None):
        """Analysis prompt template."""
        dynamic_inst = dynamic_instruction or (context.get("dynamic_instruction", "") if context else "")
        knowledge = context.get("domain_knowledge", "") if context else ""
        
        prompt = f"""You are an ESG analysis expert. Perform in-depth analysis (comparative, trend, etc.) on the text.

{dynamic_inst}

{knowledge}

Text:
{chunk_text}

Output format:
Instruction: [analysis task instruction]
Response: [detailed analysis result]"""
        return prompt


class LocalTemporalAnalysisExpert(LocalQwenExpertAdapter):
    """Temporal analysis expert."""
    def __init__(self, local_model, local_tokenizer):
        super().__init__("Temporal Analysis Expert", local_model, local_tokenizer)
    
    def get_prompt_template(self, chunk_text: str, context=None, dynamic_instruction=None):
        """Temporal analysis prompt template."""
        dynamic_inst = dynamic_instruction or (context.get("dynamic_instruction", "") if context else "")
        knowledge = context.get("domain_knowledge", "") if context else ""
        
        prompt = f"""You are an ESG temporal analysis expert. Analyze time trends, cross-year comparison and change patterns.

{dynamic_inst}

{knowledge}

Text:
{chunk_text}

Output format:
Instruction: [temporal analysis task instruction]
Response: [temporal analysis result and trends]"""
        return prompt


class LocalBenchmarkComparisonExpert(LocalQwenExpertAdapter):
    """Benchmark comparison expert."""
    def __init__(self, local_model, local_tokenizer):
        super().__init__("Benchmark Comparison Expert", local_model, local_tokenizer)
    
    def get_prompt_template(self, chunk_text: str, context=None, dynamic_instruction=None):
        """Benchmark comparison prompt template."""
        dynamic_inst = dynamic_instruction or (context.get("dynamic_instruction", "") if context else "")
        knowledge = context.get("domain_knowledge", "") if context else ""
        
        prompt = f"""You are an ESG benchmark comparison expert. Compare ESG performance with industry benchmarks or best practices.

{dynamic_inst}

{knowledge}

Text:
{chunk_text}

Output format:
Instruction: [comparison task instruction]
Response: [benchmark comparison result]"""
        return prompt


class LocalGreenwashingDetectionExpert(LocalQwenExpertAdapter):
    """Greenwashing detection expert."""
    def __init__(self, local_model, local_tokenizer):
        super().__init__("Greenwashing Detection Expert", local_model, local_tokenizer)
    
    def get_prompt_template(self, chunk_text: str, context=None, dynamic_instruction=None):
        """Greenwashing detection prompt template."""
        dynamic_inst = dynamic_instruction or (context.get("dynamic_instruction", "") if context else "")
        knowledge = context.get("domain_knowledge", "") if context else ""
        
        prompt = f"""You are an ESG greenwashing detection expert. Identify possible greenwashing, overstated claims or unsubstantiated commitments.

{dynamic_inst}

{knowledge}

Text:
{chunk_text}

Output format:
Instruction: [greenwashing detection task instruction]
Response: [detection result and greenwashing patterns]"""
        return prompt


class LocalStandardAlignmentExpert(LocalQwenExpertAdapter):
    """Standard alignment expert."""
    def __init__(self, local_model, local_tokenizer):
        super().__init__("Standard Alignment Expert", local_model, local_tokenizer)
    
    def get_prompt_template(self, chunk_text: str, context=None, dynamic_instruction=None):
        """Standard alignment prompt template."""
        dynamic_inst = dynamic_instruction or (context.get("dynamic_instruction", "") if context else "")
        knowledge = context.get("domain_knowledge", "") if context else ""
        
        prompt = f"""You are an ESG standard alignment expert. Identify ESG standards/frameworks and assess compliance and alignment.

{dynamic_inst}

{knowledge}

Text:
{chunk_text}

Output format:
Instruction: [standard alignment task instruction]
Response: [identified standards and compliance assessment]"""
        return prompt


class LocalConsistencyVerificationExpert(LocalQwenExpertAdapter):
    """Consistency verification expert."""
    def __init__(self, local_model, local_tokenizer):
        super().__init__("Consistency Verification Expert", local_model, local_tokenizer)
    
    def get_prompt_template(self, chunk_text: str, context=None, dynamic_instruction=None):
        """Consistency verification prompt template."""
        dynamic_inst = dynamic_instruction or (context.get("dynamic_instruction", "") if context else "")
        knowledge = context.get("domain_knowledge", "") if context else ""
        
        prompt = f"""You are an ESG consistency verification expert. Check consistency of data, claims and commitments; identify contradictions.

{dynamic_inst}

{knowledge}

Text:
{chunk_text}

Output format:
Instruction: [consistency verification task instruction]
Response: [consistency check result and contradictions found]"""
        return prompt


class LocalKnowledgeGraphExpert(LocalQwenExpertAdapter):
    """Knowledge graph expert."""
    def __init__(self, local_model, local_tokenizer):
        super().__init__("Knowledge Graph Expert", local_model, local_tokenizer)
    
    def get_prompt_template(self, chunk_text: str, context=None, dynamic_instruction=None):
        """Knowledge graph prompt template."""
        dynamic_inst = dynamic_instruction or (context.get("dynamic_instruction", "") if context else "")
        knowledge = context.get("domain_knowledge", "") if context else ""
        
        prompt = f"""You are an ESG knowledge graph expert. Extract entities, relations and graph structure to build an ESG knowledge network.

{dynamic_inst}

{knowledge}

Text:
{chunk_text}

Output format:
Instruction: [knowledge graph task instruction]
Response: [extracted entities, relations and structure]"""
        return prompt


# Create 11 expert instances
print("Creating 11 Phase 2 expert instances...")
print()

experts = {}

# Base layer (5)
print("[Base layer]")
experts["qa_expert"] = LocalQAExpert(model, tokenizer)
print("  1. QA Expert")

experts["summary_expert"] = LocalSummaryExpert(model, tokenizer)
print("  2. Summary Expert")

experts["extraction_expert"] = LocalExtractionExpert(model, tokenizer)
print("  3. Extraction Expert")

experts["classification_expert"] = LocalClassificationExpert(model, tokenizer)
print("  4. Classification Expert")

experts["analysis_expert"] = LocalAnalysisExpert(model, tokenizer)
print("  5. Analysis Expert")

print()
print("[Analysis layer]")
experts["temporal_analysis_expert"] = LocalTemporalAnalysisExpert(model, tokenizer)
print("  6. Temporal Analysis Expert")

experts["benchmark_comparison_expert"] = LocalBenchmarkComparisonExpert(model, tokenizer)
print("  7. Benchmark Comparison Expert")

experts["greenwashing_detection_expert"] = LocalGreenwashingDetectionExpert(model, tokenizer)
print("  8. Greenwashing Detection Expert")

print()
print("[Verification layer]")
experts["standard_alignment_expert"] = LocalStandardAlignmentExpert(model, tokenizer)
print("  9. Standard Alignment Expert")

experts["consistency_verification_expert"] = LocalConsistencyVerificationExpert(model, tokenizer)
print(" 10. Consistency Verification Expert")

print()
print("[Graph layer]")
experts["knowledge_graph_expert"] = LocalKnowledgeGraphExpert(model, tokenizer)
print(" 11. Knowledge Graph Expert")

print()
print("=" * 80)
print(f"Created {len(experts)} expert instances (Phase 2)")
print("=" * 80)


Initialize 11 Phase 2 expert instances

Creating 11 Phase 2 expert instances...

[Base layer]
  1. QA Expert
  2. Summary Expert
  3. Extraction Expert
  4. Classification Expert
  5. Analysis Expert

[Analysis layer]
  6. Temporal Analysis Expert
  7. Benchmark Comparison Expert
  8. Greenwashing Detection Expert

[Verification layer]
  9. Standard Alignment Expert
 10. Consistency Verification Expert

[Graph layer]
 11. Knowledge Graph Expert

Created 11 expert instances (Phase 2)


### 2.2 Initialize other components (ExpertSelector, DomainKnowledgeInjector, COEFramework)


In [5]:
from instruction_generation.expert_selector_02_01 import ExpertSelector
from instruction_generation.domain_knowledge_02_02 import DomainKnowledgeInjector
from instruction_generation.coe_framework_02_03 import COEFramework, ExpertOutput
from pathlib import Path

print("=" * 80)
print("Initialize system components")
print("=" * 80)
print()

# 1. Expert selector
print("[1. ExpertSelector]")
expert_selector = ExpertSelector(
    min_experts=1,
    max_experts=3,
    use_layered_selection=True,
    layered_config={
        "base_layer_required": True,
        "analysis_layer_threshold": 0.6,
        "verification_layer_threshold": 0.7,
        "graph_layer_threshold": 0.8
    }
)
print(f"  ExpertSelector initialized")
print(f"    - Expert count: {expert_selector.min_experts}-{expert_selector.max_experts}")
print(f"    - Layered: {'on' if expert_selector.use_layered_selection else 'off'}")
print(f"    - Expert types: {len(expert_selector.EXPERT_TYPES)}")
print()

# 2. Knowledge injector
print("[2. DomainKnowledgeInjector]")
knowledge_base_path = Path("/root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/knowledge_base")
knowledge_injector = DomainKnowledgeInjector(
    knowledge_base_path=str(knowledge_base_path) if knowledge_base_path.exists() else None,
    max_knowledge_items=5
)
print(f"  DomainKnowledgeInjector initialized")
if knowledge_base_path.exists():
    print(f"    - Knowledge base: {knowledge_base_path}")
else:
    print(f"    - Using built-in ESG keywords")
print(f"    - Max items: {knowledge_injector.max_knowledge_items}")
print()

# 3. CoDE Framework
print("[3. COEFramework]")
coe_framework = COEFramework(
    experts=experts,
    enable_collaboration=True,
    collaboration_mode="sequential",
    max_iterations=2,
    quality_threshold=0.7
)
print(f"  COEFramework initialized")
print(f"    - Collaboration: {'on' if coe_framework.enable_collaboration else 'off'}")
print(f"    - Mode: {coe_framework.collaboration_mode}")
print(f"    - Max iterations: {coe_framework.max_iterations}")
print(f"    - Quality threshold: {coe_framework.quality_threshold}")
print()

print("=" * 80)
print("All components initialized")
print("=" * 80)


[instruction_generation.expert_selector_02_01|WARNING] Classifier not trained; using rule-based fallback
[instruction_generation.domain_knowledge_02_02|WARNING] Knowledge base path not found; using built-in keyword knowledge


Initialize other system components

[1. Expert selector]
  ExpertSelector initialized
    - Expert count range: 1-3
    - Layered selection: enabled
    - Available experts: 12

[2. Domain knowledge injector]
  KnowledgeInjector initialized
    - Using built-in ESG keyword knowledge
    - Max knowledge items: 5

[3. CoE framework]
  CoE Framework initialized
    - Expert collaboration: enabled
    - Mode: sequential
    - Max iterations: 2
    - Quality threshold: 0.7

All components initialized


## 3. Meta-Expert implementation

Core innovation of the paper: two-stage generation using local Qwen2-72B.


In [6]:
import torch

class LocalMetaExpert:
    """
    Meta-Expert - runs on local Qwen2-72B
    
    Core innovation: two-stage generation
    1. Meta-Expert generates a personalised instruction per expert type and chunk.
    2. Each Expert uses that instruction + chunk to produce the final QA pair.
    
    This dynamic instruction generation is the key novelty of VeritasCarbon.
    """
    
    def __init__(self, model, tokenizer, is_carbon_focused=True):
        """
        Initialise MetaExpert.
        
        Args:
            model: local Qwen2-72B model
            tokenizer: tokenizer
            is_carbon_focused: whether to enforce the carbon-centric constraint
        """
        self.model = model
        self.tokenizer = tokenizer
        self.is_carbon_focused = is_carbon_focused
        
        # Expert type descriptions (aligned with expert_selector)
        self.expert_descriptions = {
            "qa_expert": "QA expert - generates high-quality question-answer pairs",
            "summary_expert": "Summary expert - produces concise text summaries",
            "extraction_expert": "Extraction expert - extracts ESG metrics and data",
            "classification_expert": "Classification expert - determines ESG dimensions and categories",
            "analysis_expert": "Analysis expert - performs baseline ESG analysis",
            "temporal_analysis_expert": "Temporal analysis expert - cross-year comparison and trend analysis",
            "benchmark_comparison_expert": "Benchmark comparison expert - compares ESG performance to industry benchmarks",
            "greenwashing_detection_expert": "Greenwashing detection expert - identifies greenwashing patterns and false commitments",
            "standard_alignment_expert": "Standard alignment expert - identifies ESG standards and assesses compliance",
            "knowledge_graph_expert": "Knowledge graph expert - extracts entities and relations",
            "consistency_verification_expert": "Consistency verification expert - detects data inconsistencies"
        }
    
    def extract_core_topics(self, chunk_text, max_topics=5):
        """
        Extract core topics from a chunk via keyword matching.
        Can be extended with NER or topic models.
        """
        esg_keywords = [
            # Chinese ESG keywords (corpus is in Chinese; keep for matching)
            "碳排放", "碳中和", "碳达峰", "温室气体", "GHG", "CO2",
            "环境", "社会", "治理", "ESG", "CSR",
            "可持续发展", "环保", "社会责任", "公司治理",
            "员工", "培训", "供应链", "合规", "风险",
            "创新", "质量", "安全", "能源", "水资源",
            "GRI", "SDG", "可持续", "绿色", "清洁"
        ]
        
        found_keywords = []
        text_lower = chunk_text.lower()
        for kw in esg_keywords:
            if kw.lower() in text_lower and kw not in found_keywords:
                found_keywords.append(kw)
                if len(found_keywords) >= max_topics:
                    break
        
        return found_keywords if found_keywords else ["ESG", "sustainability"]
    
    def generate_dynamic_instruction(
        self,
        expert_type,
        chunk_text,
        core_topics=None,
        selected_experts=None
    ):
        """
        Generate a dynamic expert instruction (core Meta-Expert function).
        
        Args:
            expert_type: target expert type
            chunk_text: input chunk text
            core_topics: list of core topics
            selected_experts: all selected experts for this chunk (multi-expert collaboration)
        
        Returns:
            str: dynamically generated instruction
        """
        if core_topics is None:
            core_topics = self.extract_core_topics(chunk_text)
        
        expert_desc = self.expert_descriptions.get(
            expert_type,
            "ESG analysis expert"
        )
        
        topics_str = ", ".join(core_topics) if core_topics else "ESG topics"
        
        # Build Meta-Expert prompt (with optional carbon-centric constraint)
        carbon_constraint = ""
        if self.is_carbon_focused:
            carbon_constraint = """
4. **Carbon-centric constraint**: The instruction should ultimately lead to an assessment of the company's carbon performance or commitment credibility.
   - Prioritise carbon-related content if present.
   - Otherwise, relate the ESG angle to carbon impact."""
        
        collaboration_hint = ""
        if selected_experts and len(selected_experts) > 1:
            expert_names = [self.expert_descriptions.get(e, e) for e in selected_experts]
            collaboration_hint = f"""
5. **Multi-expert collaboration**: This task involves: {', '.join(expert_names)}
   - Design the instruction specifically for "{expert_desc}".
   - Consider this expert's role in the overall analysis chain."""
        
        meta_prompt = f"""You are a senior ESG carbon-strategy analyst. Design a work instruction for a junior analyst.

# Context
- Junior analyst role: {expert_desc}
- Core topics in this chunk: {topics_str}
- Text preview (first 300 chars):
{chunk_text[:300]}...

# Your task
Generate ONE work instruction for the analyst. The instruction MUST:

1. **Be highly specific**: tightly tied to the core topics ({topics_str})

2. **Provoke deep thinking** along one or more of:
   - Comparative analysis (cross-year, vs. benchmarks)
   - Trend assessment (improving / worsening)
   - Critical thinking (greenwashing risk, data credibility)
   - Causal reasoning (why is this happening?)
   - Forward projection (based on current data)

3. **Be concise**: one clear question or task, 1-2 sentences{carbon_constraint}{collaboration_hint}

Output the instruction directly (no prefix like 'Work instruction:'):"""
        
        # Call local Qwen2-72B to generate the meta instruction
        messages = [{"role": "user", "content": meta_prompt}]
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            generated_ids = self.model.generate(
                **model_inputs,
                max_new_tokens=200,
                temperature=0.8,  # slightly higher for creativity
                top_p=0.9,
                do_sample=True
            )
        
        generated_ids = [
            output_ids[len(input_ids):] 
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        instruction = self.tokenizer.batch_decode(
            generated_ids, 
            skip_special_tokens=True
        )[0].strip()
        
        # Strip any prefix labels the model may have added
        # Strip Chinese instruction prefixes (backward compat for Chinese-formatted outputs)
        instruction = instruction.replace("工作指令：", "").replace("工作指令:", "").strip()
        instruction = instruction.replace("指令：", "").replace("指令:", "").strip()
        
        return instruction
    
    def generate_multi_expert_instructions(
        self,
        selected_experts,
        chunk_text,
        core_topics=None
    ):
        """
        Generate personalised instructions for multiple experts at once.
        
        Args:
            selected_experts: list of selected expert types
            chunk_text: input chunk text
            core_topics: pre-extracted core topics (optional)
        
        Returns:
            dict: {expert_type: dynamic_instruction}
        """
        if core_topics is None:
            core_topics = self.extract_core_topics(chunk_text)
        
        instructions = {}
        for expert_type in selected_experts:
            instruction = self.generate_dynamic_instruction(
                expert_type=expert_type,
                chunk_text=chunk_text,
                core_topics=core_topics,
                selected_experts=selected_experts
            )
            instructions[expert_type] = instruction
        
        return instructions


# Initialise Meta-Expert
print("=" * 80)
print("Initialising Meta-Expert")
print("=" * 80)
print()

meta_expert = LocalMetaExpert(
    model=model,
    tokenizer=tokenizer,
    is_carbon_focused=True
)

print("Meta-Expert initialised")
print("  - Model: Qwen2-72B-Instruct (local)")
print("  - Carbon-centric constraint: on")
print("  - Mode: two-stage generation (Meta-Expert -> Expert Agent)")
print()
print("=" * 80)


Initialize Meta-Expert

Meta-Expert initialized
  - Model: Qwen2-72B-Instruct (local)
  - Carbon-centric constraint: enabled
  - Function: two-stage generation (Meta-Expert -> Expert Agent)



## 4. Load preprocessed data


In [7]:
import json
from pathlib import Path

print("=" * 80)
print("Load pre-processed data")
print("=" * 80)
print()

data_path = Path("/root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/processed_corpus/chunks_clean_fixed.jsonl")

print(f"Data path: {data_path}")
print()
chunks = []
if data_path.exists():
    with open(data_path, "r", encoding="utf-8") as f:
        for line in f:
            chunks.append(json.loads(line))
    print(f"Loaded {len(chunks)} chunks")
    print()
    
    if chunks:
        print("[First chunk sample]")
        first_chunk = chunks[0]
        print(f"  - chunk_id: {first_chunk.get('chunk_id', 'N/A')}")
        print(f"  - text length: {len(first_chunk.get('text', first_chunk.get('chunk_text', '')))} chars")
        print(f"  - preview: {first_chunk.get('text', first_chunk.get('chunk_text', ''))[:150]}...")
else:
    print(f"Data file not found: {data_path}")

print()
print("=" * 80)


Load preprocessed data

Data path: /root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/processed_corpus/chunks_clean_fixed.jsonl

Loaded 543688 chunks

[First chunk example]
  - chunk_id: Layer1_2_<SZ Stock Exchange GEM Sustainability Report Guide No.3 Draft>_chunk_0
  - text length: 507 chars
  - preview: (Chinese ESG document text...)



## 5. Full QA generation pipeline

### Pipeline steps:
1. **Extract core topics** – Meta-Expert analyses the chunk.
2. **Expert selection** – ExpertSelector picks 1-3 experts.
3. **Generate dynamic instructions** – Meta-Expert produces a personalised instruction per expert (**core innovation**).
4. **Knowledge injection** – retrieve and inject relevant ESG domain knowledge.
5. **CoDE collaborative generation** – sequential multi-expert collaboration with feedback iterations.
6. **Quality evaluation** – auto-score; accept if above threshold.


In [8]:
def generate_qa_complete(
    chunk_text,
    chunk_id=None,
    use_meta_expert=True,
    use_knowledge=True,
    verbose=False
):
    """
    Full QA generation pipeline integrating all innovations.
    
    Args:
        chunk_text: input chunk text
        chunk_id: optional chunk identifier
        use_meta_expert: enable Meta-Expert (default True)
        use_knowledge: enable knowledge injection (default True)
        verbose: print step-by-step details
    
    Returns:
        dict: QA pair with full metadata
    """
    if verbose:
        print(f"\\n{'='*60}")
        print(f"Generating QA{' - ' + chunk_id if chunk_id else ''}")
        print(f"{'='*60}")
    
    # 1. Extract core topics
    core_topics = meta_expert.extract_core_topics(chunk_text)
    if verbose:
        print(f"\\n[Step 1: Topic extraction]")
        print(f"  Topics: {', '.join(core_topics)}")
    
    # 2. Expert selection
    selected_experts, selection_reasons = expert_selector.select_experts(chunk_text)
    expert_names = [expert_selector.EXPERT_TYPES[e]['name'] for e in selected_experts]
    if verbose:
        print(f"\\n[Step 2: Expert selection]")
        print(f"  Experts: {', '.join(expert_names)} (count={len(selected_experts)})")
        for i, (expert, reason) in enumerate(zip(selected_experts, selection_reasons), 1):
            print(f"    {i}. {expert_selector.EXPERT_TYPES[expert]['name']}: {reason}")
    
    # 3. Generate dynamic instructions via Meta-Expert
    dynamic_instructions = {}
    if use_meta_expert:
        if verbose:
            print(f"\\n[Step 3: Meta-Expert dynamic instruction generation]")
        dynamic_instructions = meta_expert.generate_multi_expert_instructions(
            selected_experts=selected_experts,
            chunk_text=chunk_text,
            core_topics=core_topics
        )
        if verbose:
            for expert_type, instruction in dynamic_instructions.items():
                expert_name = expert_selector.EXPERT_TYPES[expert_type]['name']
                print(f"  {expert_name}:")
                print(f"    instruction: {instruction}")
    
    # 4. Knowledge injection
    knowledge_items = []
    knowledge_context = ""
    if use_knowledge:
        knowledge_items = knowledge_injector.retrieve_knowledge(chunk_text)
        if knowledge_items:
            knowledge_context = knowledge_injector.inject_knowledge_to_prompt("", chunk_text, knowledge_items)
            if verbose:
                print(f"\\n[Step 4: Domain knowledge injection]")
                print(f"  Retrieved {len(knowledge_items)} knowledge items")
                for item in knowledge_items[:2]:
                    print(f"    - {item.get('content', '')[:60]}...")
    
    # 5. COEFramework collaborative generation
    if verbose:
        print(f"\\n[Step 5: COEFramework multi-expert collaboration]")
        print(f"  Mode: {coe_framework.collaboration_mode}")
        print(f"  Max iterations: {coe_framework.max_iterations}")
    
    context = {
        "dynamic_instructions": dynamic_instructions,
        "domain_knowledge": knowledge_context,
        "core_topics": core_topics
    }
    
    # Run COEFramework
    result = coe_framework.generate(
        chunk_text=chunk_text,
        selected_experts=selected_experts,
        context=context
    )
    
    if verbose:
        print(f"\\n[Step 6: Quality evaluation]")
        print(f"  Quality score: {result.quality_score:.3f}")
        print(f"  Expert: {result.expert_name}")
    
    # 6. Assemble final result
    final_result = {
        "question": result.instruction,
        "answer": result.response,
        "chunk": chunk_text,
        "chunk_id": chunk_id,
        "metadata": {
            "core_topics": core_topics,
            "selected_experts": selected_experts,
            "expert_names": expert_names,
            "selection_reasons": selection_reasons,
            "use_meta_expert": use_meta_expert,
            "dynamic_instructions": dynamic_instructions if use_meta_expert else None,
            "knowledge_items_count": len(knowledge_items),
            "quality_score": result.quality_score,
            "collaboration_mode": coe_framework.collaboration_mode,
            "max_iterations": coe_framework.max_iterations,
            "model": "Qwen2-72B-Instruct-Local",
            "framework": "VeritasCarbon-CoDE-v2.0"
        }
    }
    
    if verbose:
        print(f"\\n[Result]")
        print(f"  Question: {final_result['question']}")
        print(f"  Answer: {final_result['answer'][:200]}...")
        print(f"\\n{'='*60}")
    
    return final_result

print("=" * 80)
print("generate_qa_complete() defined (high-quality serial v3.0)")
print("=" * 80)
print()
print("Integrated features:")
print("  Meta-Expert dynamic instruction generation")
print("  11 Phase 2 experts (specialised prompts)")
print("  Adaptive expert selection (layered strategy)")
print("  COEFramework multi-expert collaboration (sequential + feedback)")
print("  Domain knowledge injection")
print("  Automatic quality evaluation (threshold=0.7)")
print()
print("Note: v3 is serial (no batch inference) to ensure stable quality.")
print("=" * 80)


QA generation function defined (high-quality serial v3.0)

Features:
  Meta-Expert dynamic instruction generation
  11 Phase 2 experts (specialized prompts)
  Adaptive expert selection (layered strategy)
  CoE Framework multi-expert collaboration (sequential + feedback)
  Domain knowledge injection
  Auto quality evaluation (threshold 0.7)

Note: v3 does NOT use batch inference, ensuring stable quality.


## 6. Single-chunk test run


In [9]:
# if chunks:
#     test_chunk = chunks[0]
#     chunk_id = test_chunk.get("chunk_id", "test_chunk_1")
#     chunk_text = test_chunk.get("text", test_chunk.get("chunk_text", ""))
    
#     print("=" * 80)
#     print("Single-chunk test generation")
#     print("=" * 80)
#     print()
#     print(f"[Test chunk info]")
#     print(f"  - chunk_id: {chunk_id}")
#     print(f"  - text length: {len(chunk_text)} chars")
#     print(f"  - preview: {chunk_text[:200]}...")
#     print()
    
#     result = generate_qa_complete(
#         chunk_text=chunk_text,
#         chunk_id=chunk_id,
#         use_meta_expert=True,
#         use_knowledge=True,
#         verbose=True
#     )
    
#     print()
#     print("=" * 80)
#     print("Test generation complete")
#     print("=" * 80)
# else:
#     print("No chunks available for testing")


## 7. Batch QA generation (with checkpoint)

### Features:
- Checkpoint resume – continue from last position after interruption
- Periodic auto-save every N items
- Live stats – success rate, quality score, average time
- tqdm progress bar
- Error isolation – one failure does not abort the whole run


In [ ]:
import json
import time
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

# ============================================================================
# 7. Batch QA generation (high-quality serial v3.0)
# - No batch inference (avoids off-topic answers and quality collapse)
# - Checkpoint-resumable
# ============================================================================

INPUT_FILE = Path("/root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/processed_corpus/chunks_sampled_20000_by_year.jsonl")

OUTPUT_DIR = Path("/root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/instructions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "qa_pairs_complete_v3.jsonl"
CHECKPOINT_FILE = OUTPUT_DIR / ".checkpoint_complete_v3.json"

MAX_CHUNKS = None
SAVE_INTERVAL = 10
# v3: success = has Q&A (quality_score >= 0 always); threshold is for stats only
QUALITY_THRESHOLD = 0.7
USE_META_EXPERT = True
USE_KNOWLEDGE = True

print("=" * 80)
print("Batch QA generation (serial v3.0 + checkpoint)")
print("=" * 80)
print()

print("[Config]")
print(f"  input:  {INPUT_FILE}")
print(f"  output: {OUTPUT_FILE}")
print(f"  checkpoint: {CHECKPOINT_FILE}")
print(f"  MAX_CHUNKS: {MAX_CHUNKS}")
print(f"  SAVE_INTERVAL: {SAVE_INTERVAL}")
print(f"  QUALITY_THRESHOLD: {QUALITY_THRESHOLD}")
print(f"  Meta-Expert: {USE_META_EXPERT}")
print(f"  Knowledge injection: {USE_KNOWLEDGE}")
print()

chunks = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

if MAX_CHUNKS:
    chunks = chunks[:MAX_CHUNKS]

print(f"Loaded {len(chunks):,} chunks")
print()

processed_ids = set()

# Counters:
# - success: Q&A present -> written to output
# - empty_or_parse_failed: Q/A empty (parse failure or wrong format)
# - exception: code exception (does not abort the run)
# - skipped_empty_chunk: chunk_text is empty

QUALITY_BINS = [
    ("[0.00,0.30)", 0.00, 0.30),
    ("[0.30,0.50)", 0.30, 0.50),
    ("[0.50,0.70)", 0.50, 0.70),
    ("[0.70,0.80)", 0.70, 0.80),
    ("[0.80,0.90)", 0.80, 0.90),
    ("[0.90,0.95)", 0.90, 0.95),
    ("[0.95,1.00]", 0.95, 1.01),
]

def _bin_quality(q: float) -> str:
    for label, lo, hi in QUALITY_BINS:
        if lo <= q < hi:
            return label
    return "unknown"

stats = {
    "total": len(chunks),
    "success": 0,
    "empty_or_parse_failed": 0,
    "exception": 0,
    "skipped_empty_chunk": 0,
    "avg_quality": 0.0,          # success samples only
    "avg_time_per_chunk": 0.0,   # average over all processed

    # Quality distribution (success samples)
    "q_ge_07": 0,
    "q_05_07": 0,
    "q_03_05": 0,
    "q_00_03": 0,

    # Legacy stats fields (still updated for compatibility)
    "quality_bins": {label: 0 for (label, _, _) in QUALITY_BINS},
    "quality_ge_07": 0,
    "quality_ge_08": 0,
    "quality_ge_09": 0,

    "start_time": datetime.now().isoformat(),
}

# Failure reason log
FAIL_LOG_FILE = OUTPUT_DIR / "qa_pairs_complete_v3_fail_reasons.jsonl"

quality_scores = []
time_records = []

if CHECKPOINT_FILE.exists():
    try:
        ckpt = json.loads(CHECKPOINT_FILE.read_text(encoding="utf-8"))
        processed_ids = set(ckpt.get("processed_chunk_ids", []))
        ckpt_stats = ckpt.get("stats", {})

        # Merge historical stats and fill missing quality bin keys
        if isinstance(ckpt_stats, dict) and ckpt_stats:
            stats.update(ckpt_stats)
            qb = stats.get("quality_bins") or {}
            for label, _, _ in QUALITY_BINS:
                qb.setdefault(label, 0)
            stats["quality_bins"] = qb
            for k in ["quality_ge_07", "quality_ge_08", "quality_ge_09"]:
                stats.setdefault(k, 0)

        print(f"Checkpoint found: {len(processed_ids)} chunks already done, resuming.")
        if isinstance(ckpt_stats, dict) and ckpt_stats:
            print(
                "  Last stats: "
                f"success={stats.get('success', 0)}, "
                f"skipped_empty_chunk={stats.get('skipped_empty_chunk', 0)}, "
                f"empty_or_parse_failed={stats.get('empty_or_parse_failed', 0)}, "
                f"exception={stats.get('exception', 0)}, "
                f"avg_quality={stats.get('avg_quality', 0):.3f}"
            )
        print()
    except Exception as e:
        print(f"Checkpoint load failed, starting fresh: {e}")
        processed_ids = set()
        print()
else:
    print("First run, no checkpoint.")
    print()

chunks_to_process = [c for c in chunks if c.get("chunk_id") not in processed_ids]
print("[Task summary]")
print(f"  to process: {len(chunks_to_process)}")
print(f"  already done: {len(processed_ids)}")
print()

if not chunks_to_process:
    print("All chunks processed!")
    raise SystemExit

start_time = time.time()

with open(OUTPUT_FILE, "a", encoding="utf-8") as f_out:
    pbar = tqdm(total=len(chunks_to_process), desc="Generating QA pairs (serial)")
    try:
        for i, chunk in enumerate(chunks_to_process, 1):
            chunk_id = chunk.get("chunk_id", f"chunk_{i}")
            chunk_text = chunk.get("text", chunk.get("chunk_text", ""))

            # Skip empty chunk
            if not chunk_text or not str(chunk_text).strip():
                processed_ids.add(chunk_id)
                stats["skipped_empty_chunk"] += 1
                # Log failure reason
                with open(FAIL_LOG_FILE, "a", encoding="utf-8") as f_fail:
                    f_fail.write(json.dumps({
                        "chunk_id": chunk_id,
                        "reason": "empty_chunk_text",
                        "quality_score": None,
                        "ts": datetime.now().isoformat(),
                    }, ensure_ascii=False) + "\n")
                pbar.update(1)
                pbar.set_postfix({
                    "succ": stats["success"],
                    "q≥0.7": stats.get("q_ge_07", 0),
                    "0.5≤q<0.7": stats.get("q_05_07", 0),
                    "0.3≤q<0.5": stats.get("q_03_05", 0),
                    "0≤q<0.3": stats.get("q_00_03", 0),
                    "empty": stats["empty_or_parse_failed"],
                    "exc": stats["exception"],
                    "skip": stats["skipped_empty_chunk"],
                    "q": f"{stats['avg_quality']:.3f}",
                    "t": f"{stats['avg_time_per_chunk']:.1f}s",
                })
                continue

            t0 = time.time()
            try:
                result = generate_qa_complete(
                    chunk_text=chunk_text,
                    chunk_id=chunk_id,
                    use_meta_expert=USE_META_EXPERT,
                    use_knowledge=USE_KNOWLEDGE,
                    verbose=False,
                )

                # Guard: result may not be a dict (e.g. parse exception / None return)
                if isinstance(result, dict):
                    q = float(result.get("metadata", {}).get("quality_score", 0.0) or 0.0)
                    has_qa = bool(result.get("question")) and bool(result.get("answer"))
                else:
                    q = 0.0
                    has_qa = False

                dt = time.time() - t0

                processed_ids.add(chunk_id)
                time_records.append(dt)

                # success criterion: Q&A must be non-empty
                if has_qa:
                    f_out.write(json.dumps(result, ensure_ascii=False) + "\n")
                    f_out.flush()
                    stats["success"] += 1
                    quality_scores.append(q)

                    # Quality bin counter (success samples)
                    b = _bin_quality(q)
                    stats["quality_bins"][b] = stats["quality_bins"].get(b, 0) + 1

                    # 4-bucket quality counts
                    if q >= 0.7:
                        stats["q_ge_07"] += 1
                    elif q >= 0.5:
                        stats["q_05_07"] += 1
                    elif q >= 0.3:
                        stats["q_03_05"] += 1
                    else:
                        stats["q_00_03"] += 1

                    # Legacy cumulative counters
                    if q >= 0.7:
                        stats["quality_ge_07"] += 1
                    if q >= 0.8:
                        stats["quality_ge_08"] += 1
                    if q >= 0.9:
                        stats["quality_ge_09"] += 1
                else:
                    stats["empty_or_parse_failed"] += 1
                    reason = "empty_question_or_answer"

                    # Log failure reason (brief preview to avoid large logs)
                    with open(FAIL_LOG_FILE, "a", encoding="utf-8") as f_fail:
                        f_fail.write(json.dumps({
                            "chunk_id": chunk_id,
                            "reason": reason,
                            "quality_score": q,
                            "question_preview": (result.get("question", "")[:120] if isinstance(result, dict) else ""),
                            "answer_preview": (result.get("answer", "")[:200] if isinstance(result, dict) else ""),
                            "ts": datetime.now().isoformat(),
                        }, ensure_ascii=False) + "\n")

                # Update rolling stats
                if quality_scores:
                    stats["avg_quality"] = sum(quality_scores) / len(quality_scores)
                if time_records:
                    stats["avg_time_per_chunk"] = sum(time_records) / len(time_records)

                # Periodic checkpoint save
                processed_count = (
                    stats["success"]
                    + stats["empty_or_parse_failed"]
                    + stats["exception"]
                    + stats["skipped_empty_chunk"]
                )
                if processed_count % SAVE_INTERVAL == 0:
                    CHECKPOINT_FILE.write_text(json.dumps({
                        "processed_chunk_ids": sorted(processed_ids),
                        "last_update": datetime.now().isoformat(),
                        "stats": stats,
                        "completed": False,
                    }, ensure_ascii=False, indent=2), encoding="utf-8")

                pbar.update(1)
                pbar.set_postfix({
                    "succ": stats["success"],
                    "q≥0.7": stats.get("q_ge_07", 0),
                    "0.5≤q<0.7": stats.get("q_05_07", 0),
                    "0.3≤q<0.5": stats.get("q_03_05", 0),
                    "0≤q<0.3": stats.get("q_00_03", 0),
                    "empty": stats["empty_or_parse_failed"],
                    "exc": stats["exception"],
                    "skip": stats["skipped_empty_chunk"],
                    "q": f"{stats['avg_quality']:.3f}",
                    "t": f"{stats['avg_time_per_chunk']:.1f}s",
                })

            except Exception as e:
                processed_ids.add(chunk_id)
                stats["exception"] += 1
                # Log exception
                with open(FAIL_LOG_FILE, "a", encoding="utf-8") as f_fail:
                    f_fail.write(json.dumps({
                        "chunk_id": chunk_id,
                        "reason": "exception",
                        "error": str(e),
                        "ts": datetime.now().isoformat(),
                    }, ensure_ascii=False) + "\n")
                pbar.update(1)
                pbar.set_postfix({
                    "succ": stats["success"],
                    "q≥0.7": stats.get("q_ge_07", 0),
                    "0.5≤q<0.7": stats.get("q_05_07", 0),
                    "0.3≤q<0.5": stats.get("q_03_05", 0),
                    "0≤q<0.3": stats.get("q_00_03", 0),
                    "empty": stats["empty_or_parse_failed"],
                    "exc": stats["exception"],
                    "skip": stats["skipped_empty_chunk"],
                    "q": f"{stats['avg_quality']:.3f}",
                    "t": f"{stats['avg_time_per_chunk']:.1f}s",
                })
                print(f"\nException processing {chunk_id}: {e}")

    except KeyboardInterrupt:
        print("\n\nInterrupted – saving checkpoint...")

    finally:
        pbar.close()
        elapsed = time.time() - start_time
        if quality_scores:
            stats["avg_quality"] = sum(quality_scores) / len(quality_scores)
        if time_records:
            stats["avg_time_per_chunk"] = sum(time_records) / len(time_records)
        stats["end_time"] = datetime.now().isoformat()
        stats["elapsed_seconds"] = elapsed

        CHECKPOINT_FILE.write_text(json.dumps({
            "processed_chunk_ids": sorted(processed_ids),
            "last_update": datetime.now().isoformat(),
            "stats": stats,
            "completed": False,
        }, ensure_ascii=False, indent=2), encoding="utf-8")

print()
print("=" * 80)
print("Generation finished (v3 serial)")
print("=" * 80)
print(f"success (has Q&A): {stats['success']}")
print(f"empty_or_parse_failed: {stats['empty_or_parse_failed']}")
print(f"exception: {stats['exception']}")
print(f"skipped_empty_chunk: {stats['skipped_empty_chunk']}")
print(f"avg quality (success): {stats['avg_quality']:.3f}")
print(f"avg time/chunk: {stats['avg_time_per_chunk']:.1f}s")
print()
print("[Quality distribution (success samples)]")
try:
    for k, v in stats.get("quality_bins", {}).items():
        print(f"  {k}: {v}")
    print(f"  >=0.7: {stats.get('quality_ge_07', 0)}")
    print(f"  >=0.8: {stats.get('quality_ge_08', 0)}")
    print(f"  >=0.9: {stats.get('quality_ge_09', 0)}")
except Exception as e:
    print(f"  (quality distribution unavailable) {e}")
print()
print(f"Output file: {OUTPUT_FILE}")
print(f"Checkpoint: {CHECKPOINT_FILE}")
print(f"Failure log: {FAIL_LOG_FILE}")
print("=" * 80)

Batch QA generation (high-quality serial v3.0 + checkpoint)

[Config]
  input: /root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/processed_corpus/chunks_sampled_20000_by_year.jsonl
  output: /root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/instructions/qa_pairs_complete_v3.jsonl
  checkpoint: /root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/instructions/.checkpoint_complete_v3.json
  MAX_CHUNKS: None
  SAVE_INTERVAL: 10
  QUALITY_THRESHOLD: 0.7
  Meta-Expert: True
  knowledge injection: True

Loaded 20,000 chunks

Checkpoint detected: 25 chunks processed, resuming from interruption
  Last stats: success=14, low_quality=0, empty_or_parse_failed=0, exception=0, avg_quality=0.734

[Task overview]
  pending: 19975
  done: 25



Generating QA pairs (serial):   0%|          | 0/19975 [00:50<?, ?it/s]



Interrupt detected, saving checkpoint...

Generation done (v3 serial)
success: 0
low_quality: 0
empty_or_parse_failed: 0
exception: 0
skipped_empty_chunk: 0
avg_quality (success only): 0.000
avg_speed: 0.0s/item
output: /root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/instructions/qa_pairs_complete_v3.jsonl
checkpoint: /root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/instructions/.checkpoint_complete_v3.json
fail log: /root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/instructions/qa_pairs_complete_v3_fail_reasons.jsonl


## 7.5 Live monitor – latest QA pairs

**Purpose**: Inspect quality and content of the most recently generated QA pairs.

**Usage**:
- Re-run this cell at any time during batch generation to see the latest results.
- Compare quality across different versions.
- Spot issues and tune parameters quickly.

In [ ]:
import json
from pathlib import Path
from datetime import datetime
import pandas as pd

# ============================================================================
# Live monitor – latest generated QA pairs
# ============================================================================

OUTPUT_FILE = Path("/root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/instructions/qa_pairs_complete_v3.jsonl")
CHECKPOINT_FILE = Path("/root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/instructions/.checkpoint_complete_v3.json")

def load_latest_results(n=20):
    """Load the latest n QA pairs, skipping malformed JSON lines."""
    if not OUTPUT_FILE.exists():
        return []
    
    results = []
    error_count = 0
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if line:
                try:
                    results.append(json.loads(line))
                except json.JSONDecodeError as e:
                    error_count += 1
                    if error_count <= 3:
                        print(f"JSON error on line {line_num}, skipping: {str(e)[:50]}")
                    continue
    
    if error_count > 3:
        print(f"Total malformed JSON lines skipped: {error_count}")
    
    return results[-n:] if len(results) > n else results

def get_checkpoint_stats():
    """Load checkpoint stats."""
    if not CHECKPOINT_FILE.exists():
        return None
    
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        return json.load(f)

def display_realtime_monitor(n_latest=10, show_examples=3):
    """
    Display live monitor stats and examples.
    
    Args:
        n_latest: number of latest QA pairs to load
        show_examples: number of examples to print
    """
    print("=" * 80)
    print("Live Monitor – Latest QA Pairs")
    print("=" * 80)
    print()
    
    # 1. Checkpoint stats
    checkpoint = get_checkpoint_stats()
    if checkpoint:
        print("[Checkpoint]")
        print(f"  last update: {checkpoint.get('last_update', 'N/A')}")
        stats = checkpoint.get('stats', {})
        success = stats.get('success', 0)
        failed = stats.get('failed', 0)
        total = stats.get('total', 0)
        
        print(f"  processed: {success} / {total}")
        
        # Avoid division by zero
        if success + failed > 0:
            success_rate = success / (success + failed) * 100
            print(f"  success rate: {success_rate:.1f}%")
        else:
            print(f"  success rate: N/A (not started)")
        
        print(f"  avg quality: {stats.get('avg_quality', 0):.3f}")
        print(f"  avg time/chunk: {stats.get('avg_time_per_chunk', 0):.1f}s")
        print()
    
    # 2. Load results
    latest_results = load_latest_results(n_latest)
    
    if not latest_results:
        print("No QA pairs generated yet.")
        return
    
    print(f"[Latest {len(latest_results)} QA pairs]")
    print()
    
    # 3. Quality stats
    quality_scores = [r['metadata']['quality_score'] for r in latest_results]
    
    print("Quality score distribution:")
    print(f"  mean:   {sum(quality_scores)/len(quality_scores):.3f}")
    print(f"  max:    {max(quality_scores):.3f}")
    print(f"  min:    {min(quality_scores):.3f}")
    print(f"  median: {sorted(quality_scores)[len(quality_scores)//2]:.3f}")
    print()
    
    # Quality segments
    excellent = sum(1 for s in quality_scores if s >= 0.8)
    good = sum(1 for s in quality_scores if 0.7 <= s < 0.8)
    medium = sum(1 for s in quality_scores if 0.5 <= s < 0.7)
    low = sum(1 for s in quality_scores if 0.3 <= s < 0.5)
    very_low = sum(1 for s in quality_scores if s < 0.3)
    
    print("Quality segments:")
    print(f"  excellent (>=0.8): {excellent} ({excellent/len(quality_scores)*100:.1f}%)")
    print(f"  good (0.7-0.8):    {good} ({good/len(quality_scores)*100:.1f}%)")
    print(f"  medium (0.5-0.7):  {medium} ({medium/len(quality_scores)*100:.1f}%)")
    print(f"  low (0.3-0.5):     {low} ({low/len(quality_scores)*100:.1f}%)")
    print(f"  very low (<0.3):   {very_low} ({very_low/len(quality_scores)*100:.1f}%)")
    print()
    
    # 4. Expert usage stats
    expert_usage = {}
    for r in latest_results:
        for expert in r['metadata'].get('expert_names', []):
            expert_usage[expert] = expert_usage.get(expert, 0) + 1
    
    print("Expert usage frequency:")
    for expert, count in sorted(expert_usage.items(), key=lambda x: x[1], reverse=True):
        print(f"  {expert}: {count} ({count/len(latest_results)*100:.1f}%)")
    print()
    
    # 5. Length stats
    question_lengths = [len(r['question']) for r in latest_results]
    answer_lengths = [len(r['answer']) for r in latest_results]
    
    print("Content length:")
    print(f"  avg question length: {sum(question_lengths)/len(question_lengths):.0f} chars")
    print(f"  avg answer length:   {sum(answer_lengths)/len(answer_lengths):.0f} chars")
    print()
    
    # 6. Show examples
    print("=" * 80)
    print(f"[Latest {min(show_examples, len(latest_results))} QA pair examples]")
    print("=" * 80)
    print()
    
    for i, result in enumerate(latest_results[-show_examples:], 1):
        print(f"Example {i}:")
        print(f"  Chunk ID: {result['chunk_id']}")
        print(f"  quality: {result['metadata']['quality_score']:.3f}")
        print(f"  experts: {', '.join(result['metadata']['expert_names'])}")
        print(f"  collab mode: {result['metadata'].get('collaboration_mode', 'N/A')}")
        print()
        
        print(f"  [Question] ({len(result['question'])} chars)")
        print(f"  {result['question']}")
        print()
        
        print(f"  [Answer] ({len(result['answer'])} chars)")
        answer_preview = result['answer'][:300] + "..." if len(result['answer']) > 300 else result['answer']
        print(f"  {answer_preview}")
        print()
        
        # Show dynamic instructions (if any)
        if result['metadata'].get('dynamic_instructions'):
            print(f"  [Meta-Expert dynamic instructions]")
            for expert_type, instruction in list(result['metadata']['dynamic_instructions'].items())[:2]:
                expert_name = expert_type
                print(f"    {expert_name}: {instruction[:100]}...")
            print()
        
        print("-" * 80)
        print()
    
    # 7. Quality trend (if enough data)
    if len(latest_results) >= 10:
        print("[Quality trend]")
        first_half = latest_results[:len(latest_results)//2]
        second_half = latest_results[len(latest_results)//2:]
        
        first_avg = sum(r['metadata']['quality_score'] for r in first_half) / len(first_half)
        second_avg = sum(r['metadata']['quality_score'] for r in second_half) / len(second_half)
        
        print(f"  first half avg quality: {first_avg:.3f}")
        print(f"  second half avg quality: {second_avg:.3f}")
        
        if second_avg > first_avg:
            print(f"  quality improving (+{(second_avg-first_avg):.3f})")
        elif second_avg < first_avg:
            print(f"  quality declining ({(second_avg-first_avg):.3f})")
        else:
            print(f"  quality stable")
        print()
    
    print("=" * 80)
    print("Tips:")
    print("  - Re-run this cell at any time to refresh results.")
    print("  - If quality drops, check whether parameters need tuning.")
    print("  - Low quality: inspect prompt, quality evaluator, and empty-text chunks.")
    print("=" * 80)

# Run live monitor
display_realtime_monitor(n_latest=20, show_examples=3)


## 8. Debug – inspect generated QA pairs

### Features:
- Statistical analysis – quality scores, expert usage, topic distribution
- Detailed view – full information for any QA pair
- Expert analysis – inspect prompts and output quality per expert
- Diagnostics – identify low-quality QA pairs and failure reasons


In [ ]:
import json
import random
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

OUTPUT_FILE = Path("/root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/instructions/qa_pairs_complete_v3.jsonl")

print("=" * 80)
print("QA pair debug analysis")
print("=" * 80)
print()

# 1. Load data
if OUTPUT_FILE.exists():
    results = []
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                results.append(json.loads(line))
    
    print(f"Loaded {len(results)} QA pairs")
    print()
    
    # 2. Overall stats
    print("=" * 80)
    print("[Overall stats]")
    print("=" * 80)
    
    quality_scores = [r["metadata"]["quality_score"] for r in results]
    print(f"Quality scores:")
    print(f"  mean:    {sum(quality_scores)/len(quality_scores):.3f}")
    print(f"  max:     {max(quality_scores):.3f}")
    print(f"  min:     {min(quality_scores):.3f}")
    print(f"  pass (>=0.7): {sum(1 for s in quality_scores if s >= 0.7)/len(quality_scores)*100:.1f}%")
    print()
    
    # Expert usage stats
    all_experts = []
    for r in results:
        if "metadata" in r and "selected_experts" in r["metadata"]:
            all_experts.extend(r["metadata"]["selected_experts"])
    
    expert_counter = Counter(all_experts)
    print(f"Expert usage stats:")
    print(f"  Total expert uses: {len(all_experts)}")
    print(f"  Avg experts per chunk: {len(all_experts)/len(results):.2f}")
    print()
    print(f"  Expert frequency:")
    for expert_type, count in expert_counter.most_common():
        percentage = count/len(results)*100
        expert_name = expert_selector.EXPERT_TYPES.get(expert_type, {}).get('name', expert_type)
        print(f"    {expert_name:20s}: {count:4d} ({percentage:5.1f}%)")
    print()
    
    # ============================================================================
    # 3. Expert quality analysis
    # ============================================================================
    print("=" * 80)
    print("[Expert output quality analysis]")
    print("=" * 80)
    
    expert_quality = defaultdict(list)
    for r in results:
        for expert_type in r["metadata"].get("selected_experts", []):
            expert_quality[expert_type].append(r["metadata"]["quality_score"])
    
    print(f"{'Expert type':20s} {'uses':>8s} {'avg quality':>11s} {'pass rate':>9s}")
    print("-" * 50)
    for expert_type, scores in sorted(expert_quality.items(), key=lambda x: sum(x[1])/len(x[1]), reverse=True):
        expert_name = expert_selector.EXPERT_TYPES.get(expert_type, {}).get('name', expert_type)
        avg_quality = sum(scores)/len(scores)
        pass_rate = sum(1 for s in scores if s >= 0.7)/len(scores)*100
        print(f"{expert_name:20s} {len(scores):8d} {avg_quality:10.3f} {pass_rate:7.1f}%")
    print()
    
    # ============================================================================
    # 4. Diagnostics - low quality QA pairs
    # ============================================================================
    print("=" * 80)
    print("[Diagnostics - low quality QA pairs (quality < 0.5)]")
    print("=" * 80)
    
    low_quality_results = [r for r in results if r["metadata"]["quality_score"] < 0.5]
    print(f"Low quality pairs: {len(low_quality_results)} ({len(low_quality_results)/len(results)*100:.1f}%)")
    
    if low_quality_results:
        print()
        print("Expert distribution in low quality pairs:")
        low_quality_experts = []
        for r in low_quality_results:
            low_quality_experts.extend(r["metadata"].get("selected_experts", []))
        low_expert_counter = Counter(low_quality_experts)
        for expert_type, count in low_expert_counter.most_common(5):
            expert_name = expert_selector.EXPERT_TYPES.get(expert_type, {}).get('name', expert_type)
            print(f"  {expert_name}: {count}")
        
        print()
        print("Example - low quality QA pair:")
        example = low_quality_results[0]
        print(f"  quality: {example['metadata']['quality_score']:.3f}")
        print(f"  experts: {', '.join(example['metadata']['expert_names'])}")
        print(f"  question: {example['question'][:100]}...")
        print(f"  answer: {example['answer'][:150]}...")
    print()
    
    # ============================================================================
    # 5. Examples
    # ============================================================================
    print("=" * 80)
    print("[Example 1: highest quality QA pair]")
    print("=" * 80)
    best_result = max(results, key=lambda x: x["metadata"]["quality_score"])
    print(f"quality: {best_result['metadata']['quality_score']:.3f}")
    print(f"experts: {', '.join(best_result['metadata']['expert_names'])}")
    if best_result['metadata'].get('core_topics'):
        print(f"core topics: {', '.join(best_result['metadata']['core_topics'])}")
    print(f"\n[Question]\n{best_result['question']}")
    print(f"\n[Answer]\n{best_result['answer'][:400]}...")
    if best_result['metadata'].get('dynamic_instructions'):
        print(f"\n[Meta-Expert dynamic instructions]")
        for expert_type, instruction in list(best_result['metadata']['dynamic_instructions'].items())[:2]:
            expert_name = expert_selector.EXPERT_TYPES.get(expert_type, {}).get('name', expert_type)
            print(f"  [{expert_name}]: {instruction}")
    print()
    
    print("=" * 80)
    print("[Example 2: multi-expert collaboration]")
    print("=" * 80)
    multi_expert_results = [r for r in results if len(r['metadata']['selected_experts']) > 1]
    if multi_expert_results:
        multi_result = random.choice(multi_expert_results)
        print(f"quality: {multi_result['metadata']['quality_score']:.3f}")
        print(f"experts selected: {len(multi_result['metadata']['selected_experts'])}")
        print(f"expert list: {', '.join(multi_result['metadata']['expert_names'])}")
        print(f"\n[Question]\n{multi_result['question']}")
        print(f"\n[Answer]\n{multi_result['answer'][:400]}...")
        if multi_result['metadata'].get('dynamic_instructions'):
            print(f"\n[Meta-Expert dynamic instructions per expert]")
            for expert_type, instruction in multi_result['metadata']['dynamic_instructions'].items():
                expert_name = expert_selector.EXPERT_TYPES.get(expert_type, {}).get('name', expert_type)
                print(f"  [{expert_name}]:")
                print(f"    {instruction}")
    else:
        print("  (no multi-expert cases in this batch)")
    print()
    
    # ============================================================================
    # 6. Interactive inspection
    # ============================================================================
    print("=" * 80)
    print("[Interactive inspection]")
    print("=" * 80)
    print("Tip: use the code below to inspect specific QA pairs:")
    print()
    print("# View the Nth QA pair (0-indexed)")
    print("# idx = 10")
    print("# qa = results[idx]")
    print("# print(f'quality: {qa[\"metadata\"][\"quality_score\"]:.3f}')")
    print("# print(f'experts: {\", \".join(qa[\"metadata\"][\"expert_names\"])}')")
    print("# print(f'question: {qa[\"question\"]}')")
    print("# print(f'answer: {qa[\"answer\"]}')")
    print()
    print("# View all QA pairs for a specific expert")
    print("# expert_type = 'greenwashing_detection_expert'")
    print("# expert_qas = [r for r in results if expert_type in r['metadata']['selected_experts']]")
    print("# print(f'Expert generated {len(expert_qas)} QA pairs')")
    print()
    
else:
    print(f"Output file not found: {OUTPUT_FILE}")
    print("Run the batch generation cell first.")

print("=" * 80)


### 8.2 Advanced debug - expert prompt and output analysis


In [ ]:
# ============================================================================
# Advanced debug: inspect expert prompt templates and actual outputs
# ============================================================================

print("=" * 80)
print("Expert prompt and output analysis")
print("=" * 80)
print()

# ============================================================================
# 1. Base prompt templates for all experts
# ============================================================================
print("[1. Base prompt templates per expert]")
print("=" * 80)
print()

expert_classes = {
    "qa_expert": LocalQAExpert,
    "summary_expert": LocalSummaryExpert,
    "extraction_expert": LocalExtractionExpert,
    "classification_expert": LocalClassificationExpert,
    "analysis_expert": LocalAnalysisExpert,
    "temporal_analysis_expert": LocalTemporalAnalysisExpert,
    "benchmark_comparison_expert": LocalBenchmarkComparisonExpert,
    "greenwashing_detection_expert": LocalGreenwashingDetectionExpert,
    "standard_alignment_expert": LocalStandardAlignmentExpert,
    "consistency_verification_expert": LocalConsistencyVerificationExpert,
    "knowledge_graph_expert": LocalKnowledgeGraphExpert,
}

# Generate sample prompt for each expert
sample_chunk = "Our company's total carbon emissions in 2023 were 1,000 tonnes, down 15% from 2022. We commit to carbon neutrality by 2030."
sample_context = {
    "dynamic_instruction": "[Meta-Expert dynamic instruction sample]",
    "domain_knowledge": "[Domain knowledge sample]"
}

for expert_type, expert_class in expert_classes.items():
    expert_name = expert_selector.EXPERT_TYPES.get(expert_type, {}).get('name', expert_type)
    print(f"[{expert_name}]")
    print("-" * 80)
    
    # Temporary expert instance just to inspect the prompt template
    try:
        temp_expert = expert_class(model, tokenizer)
        prompt = temp_expert.get_prompt_template(
            chunk_text=sample_chunk,
            context=sample_context,
            dynamic_instruction=sample_context["dynamic_instruction"]
        )
        
        # Show first 500 chars of prompt
        print(prompt[:500])
        if len(prompt) > 500:
            print("...")
        print()
    except Exception as e:
        print(f"  Cannot get prompt template: {e}")
        print()

print()

# ============================================================================
# 2. Analyze actual outputs of a specific expert
# ============================================================================
print("=" * 80)
print("[2. Analyze actual outputs of a specific expert]")
print("=" * 80)
print()

# Change ANALYZE_EXPERT to inspect a different expert
ANALYZE_EXPERT = "greenwashing_detection_expert"

if OUTPUT_FILE.exists() and 'results' in locals():
    expert_name = expert_selector.EXPERT_TYPES.get(ANALYZE_EXPERT, {}).get('name', ANALYZE_EXPERT)
    print(f"Analyzing expert: {expert_name}")
    print()
    
    # Find all QA pairs involving this expert
    expert_qas = [r for r in results if ANALYZE_EXPERT in r['metadata'].get('selected_experts', [])]
    
    if expert_qas:
        print(f"Expert involved in {len(expert_qas)} QA pairs")
        
        # Quality stats
        expert_quality_scores = [qa['metadata']['quality_score'] for qa in expert_qas]
        print(f"avg quality: {sum(expert_quality_scores)/len(expert_quality_scores):.3f}")
        print(f"max quality: {max(expert_quality_scores):.3f}")
        print(f"min quality: {min(expert_quality_scores):.3f}")
        print(f"pass rate (>=0.7): {sum(1 for s in expert_quality_scores if s >= 0.7)/len(expert_quality_scores)*100:.1f}%")
        print()
        
        # Best case
        print("-" * 80)
        print("[Best case]")
        print("-" * 80)
        best_qa = max(expert_qas, key=lambda x: x['metadata']['quality_score'])
        print(f"quality: {best_qa['metadata']['quality_score']:.3f}")
        print(f"experts: {', '.join(best_qa['metadata']['expert_names'])}")
        print()
        print(f"[Question]\n{best_qa['question']}")
        print()
        print(f"[Answer]\n{best_qa['answer'][:500]}...")
        print()
        
        if best_qa['metadata'].get('dynamic_instructions', {}).get(ANALYZE_EXPERT):
            print(f"[Meta-Expert dynamic instruction for this expert]")
            print(best_qa['metadata']['dynamic_instructions'][ANALYZE_EXPERT])
            print()
        
        # Worst case
        print("-" * 80)
        print("[Worst case (for improvement)]")
        print("-" * 80)
        worst_qa = min(expert_qas, key=lambda x: x['metadata']['quality_score'])
        print(f"quality: {worst_qa['metadata']['quality_score']:.3f}")
        print(f"experts: {', '.join(worst_qa['metadata']['expert_names'])}")
        print()
        print(f"[Question]\n{worst_qa['question']}")
        print()
        print(f"[Answer]\n{worst_qa['answer'][:500]}...")
        print()
        
        if worst_qa['metadata'].get('dynamic_instructions', {}).get(ANALYZE_EXPERT):
            print(f"[Meta-Expert dynamic instruction for this expert]")
            print(worst_qa['metadata']['dynamic_instructions'][ANALYZE_EXPERT])
            print()
        
        # Random case
        print("-" * 80)
        print("[Random case]")
        print("-" * 80)
        random_qa = random.choice(expert_qas)
        print(f"quality: {random_qa['metadata']['quality_score']:.3f}")
        print(f"experts: {', '.join(random_qa['metadata']['expert_names'])}")
        print()
        print(f"[Question]\n{random_qa['question']}")
        print()
        print(f"[Answer]\n{random_qa['answer'][:500]}...")
        print()
        
    else:
        print(f"Expert has not participated in any QA pairs yet.")
        print()
else:
    print("Run the batch generation and basic analysis cells first.")

print()
print("=" * 80)
print("Tip: change ANALYZE_EXPERT to inspect a different expert.")
print("Choices:")
for expert_type in expert_classes.keys():
    expert_name = expert_selector.EXPERT_TYPES.get(expert_type, {}).get('name', expert_type)
    print(f"  - '{expert_type}' ({expert_name})")
print("=" * 80)


## 8. View generated QA pairs and stats

In [ ]:
import json
from pathlib import Path
from collections import Counter

print("=" * 80)
print("Generated QA pairs - statistics and samples")
print("=" * 80)
print()

OUTPUT_FILE = Path("/root/autodl-fs/Veritas/VeritasCarbon_SIGMOD/data/instructions/qa_pairs_complete_v3.jsonl")

if OUTPUT_FILE.exists():
    # Load all results
    results = []
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            results.append(json.loads(line))
    
    print(f"[Basic stats]")
    print(f"  Total QA pairs: {len(results)}")
    print()
    
    if results:
        # 1. Quality stats
        quality_scores = [r["metadata"]["quality_score"] for r in results if "metadata" in r]
        print(f"[Quality stats]")
        print(f"  avg quality: {sum(quality_scores)/len(quality_scores):.3f}")
        print(f"  max quality: {max(quality_scores):.3f}")
        print(f"  min quality: {min(quality_scores):.3f}")
        print(f"  pass rate (>=0.7): {sum(1 for s in quality_scores if s >= 0.7)/len(quality_scores)*100:.1f}%")
        print()
        
        # 2. Expert usage stats
        all_experts = []
        for r in results:
            if "metadata" in r and "selected_experts" in r["metadata"]:
                all_experts.extend(r["metadata"]["selected_experts"])
        
        expert_counter = Counter(all_experts)
        print(f"[Expert usage stats]")
        print(f"  Total expert uses: {len(all_experts)}")
        print(f"  Avg experts per chunk: {len(all_experts)/len(results):.2f}")
        print()
        print(f"  Expert frequency (Top 5):")
        for expert_type, count in expert_counter.most_common(5):
            percentage = count/len(results)*100
            print(f"    - {expert_type}: {count} ({percentage:.1f}%)")
        print()
        
        # 3. Topic stats
        all_topics = []
        for r in results:
            if "metadata" in r and "core_topics" in r["metadata"]:
                all_topics.extend(r["metadata"]["core_topics"])
        
        topic_counter = Counter(all_topics)
        print(f"[Core topic stats]")
        print(f"  Distinct topics: {len(set(all_topics))}")
        print()
        print(f"  Top 10 topics:")
        for topic, count in topic_counter.most_common(10):
            percentage = count/len(results)*100
            print(f"    - {topic}: {count} ({percentage:.1f}%)")
        print()
        
        # 4. Meta-Expert usage
        use_meta_count = sum(1 for r in results if r.get("metadata", {}).get("use_meta_expert", False))
        print(f"[Meta-Expert usage]")
        print(f"  Used Meta-Expert: {use_meta_count} ({use_meta_count/len(results)*100:.1f}%)")
        print()
        
        # 5. Examples
        print("=" * 80)
        print("[Example 1: highest quality QA pair]")
        print("=" * 80)
        best_result = max(results, key=lambda x: x["metadata"]["quality_score"])
        print(f"\\nquality: {best_result['metadata']['quality_score']:.3f}")
        print(f"experts: {', '.join(best_result['metadata']['expert_names'])}")
        print(f"core topics: {', '.join(best_result['metadata']['core_topics'])}")
        print(f"\\nQuestion: {best_result['question']}")
        print(f"\\nAnswer: {best_result['answer'][:300]}...")
        if best_result['metadata'].get('dynamic_instructions'):
            print(f"\\nMeta-Expert dynamic instructions:")
            for expert_type, instruction in list(best_result['metadata']['dynamic_instructions'].items())[:1]:
                print(f"  [{expert_type}]: {instruction}")
        
        print()
        print("=" * 80)
        print("[Example 2: random QA pair]")
        print("=" * 80)
        import random
        random_result = random.choice(results)
        print(f"\\nquality: {random_result['metadata']['quality_score']:.3f}")
        print(f"experts: {', '.join(random_result['metadata']['expert_names'])}")
        print(f"core topics: {', '.join(random_result['metadata']['core_topics'])}")
        print(f"\\nQuestion: {random_result['question']}")
        print(f"\\nAnswer: {random_result['answer'][:300]}...")
        
        print()
        print("=" * 80)
        print("[Example 3: multi-expert collaboration]")
        print("=" * 80)
        # Find a multi-expert example
        multi_expert_results = [r for r in results if len(r['metadata']['selected_experts']) > 1]
        if multi_expert_results:
            multi_result = random.choice(multi_expert_results)
            print(f"\\nquality: {multi_result['metadata']['quality_score']:.3f}")
            print(f"experts selected: {len(multi_result['metadata']['selected_experts'])}")
            print(f"expert list: {', '.join(multi_result['metadata']['expert_names'])}")
            print(f"core topics: {', '.join(multi_result['metadata']['core_topics'])}")
            print(f"\\nQuestion: {multi_result['question']}")
            print(f"\\nAnswer: {multi_result['answer'][:250]}...")
            if multi_result['metadata'].get('dynamic_instructions'):
                print(f"\\nMeta-Expert dynamic instructions per expert:")
                for expert_type, instruction in multi_result['metadata']['dynamic_instructions'].items():
                    expert_name = expert_selector.EXPERT_TYPES.get(expert_type, {}).get('name', expert_type)
                    print(f"  [{expert_name}]: {instruction}")
        else:
            print("  (no multi-expert cases in this batch)")
        
else:
    print(f"Output file not found: {OUTPUT_FILE}")
    print("Run the batch generation cell first.")

print()
print("=" * 80)
